# Issue #225 — Normal-junction filter strength on patient_001 (chr22)

**Issue:** [#225](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/225)  
**Parent:** [Issue #203](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/203) (Experiment 3)  
**Design spec:** [docs/superpowers/specs/2026-05-21-issue-225-normal-junction-filter-strength-design.md](../../../docs/superpowers/specs/2026-05-21-issue-225-normal-junction-filter-strength-design.md)

Compare three normal-junction filter sources on patient_001's chr22 tumor junctions: matched-normal, Snaptron-chr22-GTEx-proxy, and AlphaGenome-predicted-normal. Produce the decision-rule numbers for the Exp 3 row of [Issue #203](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/203)'s decision table.

**Scope:** chr22 only (test-config harness); patient_001 only; matching key `(chrom, donor_0based, acceptor_0based_excl, strand)`, exact match.

## §0 — Setup + path guards

In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "workflow").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENT_DIR = REPO_ROOT / "research" / "experiments" / "issue_225_normal_junction_filter_strength"
OUTPUTS_DIR = EXPERIMENT_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CHROM = "chr22"

# Inputs (referenced by path; not copied)
TUMOR_TSV = REPO_ROOT / "results" / "patient_001_test" / "alignment" / "SRR9143066_test" / "junctions.tsv"
NORMAL_TSV = REPO_ROOT / "results" / "patient_001_test" / "alignment" / "SRR9143065_test" / "junctions.tsv"
AG_PARQUET = REPO_ROOT / "research" / "notebooks" / "issue_224_alphagenome_exp1_outputs" / "chr22_stomach_predicted_junctions.parquet"

# Outputs
GTEX_PANEL_PARQUET = OUTPUTS_DIR / "chr22_gtex_panel.parquet"
OVERLAP_TSV = OUTPUTS_DIR / "filter_overlap_table.tsv"
VENN_PNG = OUTPUTS_DIR / "filter_venn_chr22.png"

# Path-existence guards
inputs = {
    "Tumor junctions": TUMOR_TSV,
    "Matched-normal junctions": NORMAL_TSV,
    "AlphaGenome predictions": AG_PARQUET,
}
missing = [(label, path) for label, path in inputs.items() if not path.exists()]
if missing:
    msg_lines = [f"  - {label}: {path}" for label, path in missing]
    fix_hint = (
        "\nTo produce the chr22 tumor junctions (the most common miss), run:\n"
        "  conda activate snakemake && snakemake --cores 4 --use-conda --configfile config/test_config.yaml "
        "-- results/patient_001_test/alignment/SRR9143066_test/junctions.tsv"
    )
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(msg_lines) + fix_hint)

print(f"Repo root:      {REPO_ROOT}")
print(f"Experiment dir: {EXPERIMENT_DIR.relative_to(REPO_ROOT)}")
print(f"All inputs present: {all(p.exists() for p in inputs.values())}")

## §1 — Tumor junctions (patient_001 chr22)

Load the post-PR #372 `junctions.tsv` produced by `workflow/scripts/bed12_to_junctions.py`. Format: 2-column TSV, no header: `<chrom>:<donor_1based>:<acceptor_0based_excl>:<strand>\t<reads>`. Loader normalizes to **0-based half-open intron coords** for set-ops downstream.

In [ ]:
def load_pipeline_junctions(tsv_path: Path) -> pd.DataFrame:
    """Load 2-column junctions TSV from bed12_to_junctions.py / STAR awk.

    Format: ``<chrom>:<donor_1based>:<acceptor_0based_excl>:<strand>\\t<reads>``.
    Returns DataFrame with 0-based half-open intron coords for set operations.
    """
    raw = pd.read_csv(tsv_path, sep="\t", header=None, names=["key", "count"])
    parts = raw["key"].str.split(":", expand=True)
    parts.columns = ["chrom", "donor_1based", "acceptor_0based_excl", "strand"]
    return pd.DataFrame({
        "chrom": parts["chrom"],
        "donor": parts["donor_1based"].astype(int) - 1,
        "acceptor": parts["acceptor_0based_excl"].astype(int),
        "strand": parts["strand"],
        "count": raw["count"].astype(int),
    })

tumor = load_pipeline_junctions(TUMOR_TSV)
assert (tumor["chrom"] == TARGET_CHROM).all(), f"Expected {TARGET_CHROM}-only tumor; got {tumor['chrom'].unique()}"
assert len(tumor) > 0, "Tumor junctions empty — check alignment for SRR9143066"
print(f"Tumor junctions (chr22):     {len(tumor):,}")
print(f"Read-count summary: {tumor['count'].describe().to_dict()}")
tumor.head()

## §2 — Filter sets

### §2(a) — Matched-normal (current pipeline source)

Load matched-normal junctions from the same `junctions.tsv` format. Anchor of the comparison; gold standard for tissue-expressed splicing in patient_001's stomach normal.

In [ ]:
mn = load_pipeline_junctions(NORMAL_TSV)
assert (mn["chrom"] == TARGET_CHROM).all()
assert len(mn) > 0

# Build the matching-key set: (chrom, donor, acceptor, strand)
def to_key_set(df: pd.DataFrame) -> set[tuple]:
    return set(map(tuple, df[["chrom", "donor", "acceptor", "strand"]].itertuples(index=False, name=None)))

mn_set = to_key_set(mn)
print(f"Matched-normal junctions:    {len(mn):,}  (unique keys: {len(mn_set):,})")

### §2(b) — AlphaGenome predicted-normal

Load AG predictions cached by [#224](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/224) at `research/notebooks/issue_224_alphagenome_exp1_outputs/chr22_stomach_predicted_junctions.parquet` *(cross-experiment dep)*. Recompute the F1-max threshold against the ground-truth definition from #224 §5 (matched-normal ∩ GENCODE-annotated, both restricted to chr22). Threshold-binarise to produce `ag_set`.

In [ ]:
import gzip
import re

# Load AG predictions
ag = pd.read_parquet(AG_PARQUET)
assert {"chrom", "donor", "acceptor", "strand"}.issubset(ag.columns), f"AG parquet missing key cols: {ag.columns.tolist()}"
score_col = "score" if "score" in ag.columns else next((c for c in ag.columns if "score" in c.lower() or "prob" in c.lower()), None)
assert score_col is not None, f"No score column found in AG parquet; columns: {ag.columns.tolist()}"
print(f"AG predictions: {len(ag):,}  | score column: {score_col}")

# Build the ground-truth set for F1: matched-normal ∩ GENCODE-annotated (chr22)
GENCODE_GTF = REPO_ROOT / "resources" / "test" / "chr22.gtf.gz"
assert GENCODE_GTF.exists(), f"GENCODE chr22 GTF not found: {GENCODE_GTF}"

def gencode_introns_chr22(gtf_path: Path) -> set[tuple]:
    """Return set of (chrom, donor_0based, acceptor_0based_excl, strand) introns from a GENCODE GTF.

    Introns are inferred from consecutive exons within the same transcript.
    """
    transcripts: dict[str, list] = {}
    with gzip.open(gtf_path, "rt") as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            f = line.rstrip("\n").split("\t")
            if len(f) < 9 or f[2] != "exon":
                continue
            chrom, start, end, strand, attrs = f[0], int(f[3]) - 1, int(f[4]), f[6], f[8]
            m = re.search(r'transcript_id "([^"]+)"', attrs)
            if not m:
                continue
            transcripts.setdefault(m.group(1), []).append((chrom, start, end, strand))
    introns = set()
    for txid, exons in transcripts.items():
        exons.sort(key=lambda e: e[1])
        for a, b in zip(exons, exons[1:]):
            if a[0] != b[0] or a[3] != b[3]:
                continue
            introns.add((a[0], a[2], b[1], a[3]))  # (chrom, donor=prev_exon_end, acceptor=next_exon_start, strand)
    return introns

gencode_introns = gencode_introns_chr22(GENCODE_GTF)
print(f"GENCODE chr22 introns:       {len(gencode_introns):,}")

ground_truth = mn_set & gencode_introns
print(f"Ground truth (MN ∩ GENCODE): {len(ground_truth):,}")
assert len(ground_truth) > 0, "Empty ground truth — chr22 GENCODE intersection failed; check coord conventions"

In [ ]:
from sklearn.metrics import precision_recall_curve

# Universe-restricted F1-max sweep — matches #224 §5 semantics so the F1 number
# below is directly comparable to #224's, and to the τ-thresholds in the §6
# decision rule (which were calibrated on that universe-restricted F1).
#   universe = GENCODE chr22 introns        (n=7,731 expected)
#   label    = 1 if intron ∈ matched-normal (positives = MN ∩ GENCODE, n=259 expected)
#   score    = max AG score for that intron's key; 0 if not present in AG
ag_dedup_by_key = (
    ag.groupby(["chrom", "donor", "acceptor", "strand"], as_index=False)[score_col].max()
)
ag_score_lookup = dict(
    zip(
        map(tuple, ag_dedup_by_key[["chrom", "donor", "acceptor", "strand"]].itertuples(index=False, name=None)),
        ag_dedup_by_key[score_col].to_numpy(),
    )
)

universe_keys = list(gencode_introns)
universe_scores = np.array([ag_score_lookup.get(k, 0.0) for k in universe_keys], dtype=float)
universe_labels = np.array([1 if k in mn_set else 0 for k in universe_keys], dtype=np.int8)
n_pos = int(universe_labels.sum())
n_neg = len(universe_labels) - n_pos
print(f"Universe (GENCODE chr22 introns): {len(universe_keys):,}")
print(f"  Positives (in matched-normal):  {n_pos:,}")
print(f"  Negatives:                       {n_neg:,}")
print(f"  AG-scored (score > 0):           {(universe_scores > 0).sum():,}")

precision, recall, thresholds = precision_recall_curve(universe_labels, universe_scores)
denom = precision + recall
f1_arr = np.where(denom > 0, 2 * precision * recall / np.where(denom > 0, denom, 1.0), 0.0)
# sklearn appends a sentinel (P=1, R=0) point with no corresponding threshold;
# argmax over f1_arr[:-1] keeps the indices aligned with `thresholds`.
best_idx = int(np.argmax(f1_arr[:-1]))
best_threshold = float(thresholds[best_idx])
best_precision = float(precision[best_idx])
best_recall = float(recall[best_idx])
best_f1 = float(f1_arr[best_idx])
print(
    f"Best F1: {best_f1:.4f} at τ = {best_threshold:.6f}  "
    f"(P={best_precision:.3f}, R={best_recall:.3f})"
)

best = pd.Series({
    "threshold": best_threshold,
    "precision": best_precision,
    "recall": best_recall,
    "f1": best_f1,
})

In [ ]:
ag_predicted_normal = ag[ag[score_col] >= best_threshold]
ag_set = to_key_set(ag_predicted_normal)
print(f"AG predicted-normal @ τ={best_threshold:.4f}: {len(ag_set):,} junctions")

### §2(c) — Snaptron chr22 GTEx pan-tissue proxy

Build a chr22 pan-tissue GTEx union by querying Snaptron's hg38 GTEx v2 endpoint. Inclusion: junction observed in ≥ 1 GTEx sample across any tissue (most-conservative pan-tissue; matches #203's vaccine-safety reasoning).

**Caveat:** this is a proxy for the production GTEx panel built by [#211](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/211). When #211 lands, re-run this cell against the production panel; only the GTEx-set construction changes downstream.

Result is cached to `outputs/chr22_gtex_panel.parquet`; subsequent runs load the cache.

In [ ]:
import io
import time
import urllib.request
import urllib.error

SNAPTRON_GTEX_URL = "https://snaptron.cs.jhu.edu/gtexv2/snaptron"
SNAPTRON_CHR22_REGION = "chr22:1-50818468"

def fetch_snaptron_chr22(region: str = SNAPTRON_CHR22_REGION, timeout_s: int = 180, retry: bool = True) -> pd.DataFrame:
    """Query Snaptron GTEx v2 for chr22 junctions. Returns parsed DataFrame."""
    url = f"{SNAPTRON_GTEX_URL}?regions={region}"
    attempts = 2 if retry else 1
    last_err = None
    for attempt in range(1, attempts + 1):
        try:
            print(f"Snaptron query (attempt {attempt}/{attempts}): {url}")
            with urllib.request.urlopen(url, timeout=timeout_s) as resp:
                raw = resp.read().decode("utf-8")
            break
        except urllib.error.URLError as e:
            last_err = e
            if attempt < attempts:
                time.sleep(5)
            else:
                raise
    df = pd.read_csv(io.StringIO(raw), sep="\t")
    print(f"Snaptron rows returned: {len(df):,}")
    return df

def snaptron_to_key_set(df: pd.DataFrame, min_samples: int = 1) -> set[tuple]:
    """Convert Snaptron DataFrame to the (chrom, donor, acceptor, strand) key set.

    Snaptron GTEx v2 cols: chromosome, start, end, strand, samples_count.
    `start` is 1-based inclusive intron donor; `end` is 1-based inclusive acceptor
    (= 0-based exclusive acceptor). Normalise to 0-based half-open intron coords.
    """
    needed = {"chromosome", "start", "end", "strand", "samples_count"}
    assert needed.issubset(df.columns), f"Snaptron schema mismatch; got {df.columns.tolist()}"
    df = df[df["samples_count"] >= min_samples]
    df = df[df["chromosome"] == TARGET_CHROM]
    keys = set()
    starts = df["start"].astype(int).values
    ends = df["end"].astype(int).values
    chroms = df["chromosome"].values
    strands = df["strand"].values
    for chrom, start, end, strand in zip(chroms, starts, ends, strands):
        keys.add((chrom, int(start) - 1, int(end), strand))
    return keys

if GTEX_PANEL_PARQUET.exists():
    print(f"Loading cached GTEx panel from {GTEX_PANEL_PARQUET}")
    gtex_panel = pd.read_parquet(GTEX_PANEL_PARQUET)
else:
    raw_gtex = fetch_snaptron_chr22()
    gtex_panel = raw_gtex[["chromosome", "start", "end", "strand", "samples_count"]].copy()
    assert (gtex_panel["chromosome"] == TARGET_CHROM).all(), "Snaptron returned non-chr22 rows"
    assert len(gtex_panel) > 0, "Empty Snaptron result"
    gtex_panel.to_parquet(GTEX_PANEL_PARQUET, index=False)
    print(f"Cached GTEx panel to {GTEX_PANEL_PARQUET}  ({len(gtex_panel):,} rows)")

gtex_set = snaptron_to_key_set(gtex_panel, min_samples=1)
print(f"GTEx chr22 pan-tissue (≥1 sample): {len(gtex_set):,} unique junction keys")

## §3 — Apply each filter separately to tumor junctions

For each filter source F, compute `tumor_caught_by_F = tumor ∩ F`. The filter "catches" tumor junctions that also appear in F — these would be removed as non-tumor-specific.

In [ ]:
tumor_set = to_key_set(tumor)
assert len(tumor_set) > 0

caught_by_mn   = tumor_set & mn_set
caught_by_gtex = tumor_set & gtex_set
caught_by_ag   = tumor_set & ag_set

# Sanity guard: each caught set is a subset of tumor
for label, caught in [("MN", caught_by_mn), ("GTEx", caught_by_gtex), ("AG", caught_by_ag)]:
    assert caught.issubset(tumor_set), f"{label} caught set is not a subset of tumor — coord-system mismatch?"
    pct = 100 * len(caught) / len(tumor_set)
    print(f"Tumor caught by {label:5s}: {len(caught):6,}  ({pct:5.1f}% of tumor)")
print(f"Total tumor junctions:       {len(tumor_set):6,}")

## §4 — Overlap analysis

Pairwise + triple intersections of the three caught sets; unique contributions (caught by exactly one source). Output exported as `outputs/filter_overlap_table.tsv` for the #203 decision-rule writeup.

In [ ]:
# Pairwise + triple intersections
mn_gtex   = caught_by_mn & caught_by_gtex
mn_ag     = caught_by_mn & caught_by_ag
gtex_ag   = caught_by_gtex & caught_by_ag
all_three = caught_by_mn & caught_by_gtex & caught_by_ag

# Unique contributions (caught by exactly one)
only_mn   = caught_by_mn - caught_by_gtex - caught_by_ag
only_gtex = caught_by_gtex - caught_by_mn - caught_by_ag
only_ag   = caught_by_ag - caught_by_mn - caught_by_gtex

# Net stacked filter (union)
caught_by_any = caught_by_mn | caught_by_gtex | caught_by_ag

overlap_table = pd.DataFrame([
    {"region": "MN",                       "n": len(caught_by_mn)},
    {"region": "GTEx",                     "n": len(caught_by_gtex)},
    {"region": "AG",                       "n": len(caught_by_ag)},
    {"region": "MN ∩ GTEx",                "n": len(mn_gtex)},
    {"region": "MN ∩ AG",                  "n": len(mn_ag)},
    {"region": "GTEx ∩ AG",                "n": len(gtex_ag)},
    {"region": "MN ∩ GTEx ∩ AG",           "n": len(all_three)},
    {"region": "Only MN",                  "n": len(only_mn)},
    {"region": "Only GTEx",                "n": len(only_gtex)},
    {"region": "Only AG",                  "n": len(only_ag)},
    {"region": "Caught by any (union)",    "n": len(caught_by_any)},
    {"region": "Tumor total",              "n": len(tumor_set)},
])
overlap_table["pct_of_tumor"] = (100 * overlap_table["n"] / len(tumor_set)).round(2)

overlap_table.to_csv(OVERLAP_TSV, sep="\t", index=False)
print(f"Wrote {OVERLAP_TSV}")
overlap_table

In [ ]:
# Read-back check: serialised TSV must match the in-memory table
read_back = pd.read_csv(OVERLAP_TSV, sep="\t")
pd.testing.assert_frame_equal(read_back, overlap_table, check_dtype=False)
print("Read-back check passed.")